### 1 - Data Ingesion & Integration

In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [11]:
#Dynamic Path Detection so it runs on all user environments
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")

In [12]:
#Spark Initialization
spark = SparkSession.builder \
    .appName("BODS_Predictive_Engine") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [13]:
# Load files
df_cat = spark.read.csv(os.path.join(DATA_DIR, "overall_data_catalogue.csv"), header=True, inferSchema=True)
df_comp = spark.read.csv(os.path.join(DATA_DIR, "overall_compliance_report.csv"), header=True, inferSchema=True)

# Surgical Join: Joining on Service Code (Catalogue) and Service Number (Compliance) <Multi Catalogue Ingestion>
df_unified = df_cat.join(
    df_comp, 
    df_cat["Service Code"] == df_comp["Registration:Service Number"], 
    how="right"
)

print(f"Row count after surgical join: {df_unified.count()}")

[Stage 43:=========>        (2 + 2) / 4][Stage 44:>                 (0 + 2) / 2]

Row count after surgical join: 103437


### 2 - Data Cleaning

In [14]:
# 1. Drop rows where critical compliance data is missing
df_cleaned = df_unified.fillna({'Service Code': 'UNKNOWN', 'Status': 'UNKNOWN'})

# 2. Impute missing numeric values (replace NULLs with 0)
df_cleaned = df_cleaned.fillna(0, subset=["% AVL to Timetables feed matching score"])

# 3. Final validation of record count
final_count = df_cleaned.count()
print(f"### **Final Cleaned Record Count: {final_count}**")

[Stage 52:=========>        (2 + 2) / 4][Stage 53:=========>        (1 + 1) / 2]

### **Final Cleaned Record Count: 103437**


In [32]:
# PySpark optimization
#Repartitioning to match cluster configuration and optimizing down to a safe state

df_optimized = df_cleaned.repartition(4)

#In-memory caching to optimize downstream aggregations and disk-writes
df_optimized.cache()

print(f"Cached partition count: {df_optimized.rdd.getNumPartitions()}")

[Stage 89:>                                                         (0 + 4) / 4]

Cached partition count: 4


[Stage 89:=============================>                            (2 + 2) / 4]

### PySpark SQL Integration: Complex Aggregations


In [36]:
df_optimized.createOrReplaceTempView("v_compliance_ledger")

# Execute analytical query calculating average compliance scores by Operator
compliance_summary_df = spark.sql("""
    SELECT 
        `Operator`,
        COUNT(*) as total_services,
        ROUND(AVG(CAST(`% AVL to Timetables feed matching score` AS DOUBLE)), 2) as avg_matching_score
    FROM v_compliance_ledger
    GROUP BY `Operator`
    HAVING total_services > 100
    ORDER BY total_services DESC
""")

# Show the analytical output
compliance_summary_df.show(5, truncate=False)

+-------------------------+--------------+------------------+
|Operator                 |total_services|avg_matching_score|
+-------------------------+--------------+------------------+
|Arriva UK Bus            |42887         |0.0               |
|Go-Ahead                 |23029         |0.0               |
|Ipswich Buses            |8734          |0.0               |
|NULL                     |7934          |0.0               |
|Nottingham City Transport|2514          |0.0               |
+-------------------------+--------------+------------------+
only showing top 5 rows



### 3 - Data Storage & Processing

In [15]:
import sqlite3
import pandas as pd

In [20]:
# 1. Intermediate Write using optimized source ptr
intermediate_path = os.path.join(DATA_DIR, "cleaned_checkpoint_csv")
df_cleaned.write.mode("overwrite").option("header", "true").csv(intermediate_path)

In [23]:
# 2. Loading Relational Database
db_path = os.path.join(DATA_DIR, "bods_analytics.db")
df_checkpoint = spark.read.option("header", "true").csv(intermediate_path)

In [24]:
# Conversion to Pandas for native SQLite loading
pandas_df = df_checkpoint.toPandas()
conn = sqlite3.connect(db_path)
pandas_df.to_sql("operator_compliance", conn, if_exists="replace", index=False)

103437

In [31]:
# #3. Secure Query Interface (Writing parameterized queries to prevent SQL injection)
import os

# Open a fresh connection so the function can read the database file
db_path = os.path.join(DATA_DIR, "bods_analytics.db")
conn = sqlite3.connect(db_path)

def secure_search_operator(operator_name):
    query = 'SELECT * FROM operator_compliance WHERE "Operator" = ? LIMIT 5'
    cursor = conn.cursor()
    cursor.execute(query, (operator_name,))
    results = cursor.fetchall()  # Fetches the rows
    cursor.close()
    return results               # Passes the rows back to the variable

# Run the test query
test_results = secure_search_operator("Go-Ahead")
print(test_results)

# Safely close the connection after printing
conn.close()

[('Go-Ahead', '10.0', 'BHBC; BLUS; BTRI; CROP; CSLB; CSSO; DAMY; DBLU; DDIS; ETBU; EYMS; FAST; GAGL; GAHL; GAWY; GNEL; GONW; GOSW; KNCO; LCAG; LGEN; LONC; MBGA; METR; NATI; OEXP; OXBC; PLYC; PROC; PULH; SDMS; SVCT; SWWD; TDTR; TFCN; THTR; TOUR; UNBU; UNIL; WDBC', 'Timetables', 'inactive', '2021-09-30T22:07:48.350+05:45', 'remote_dataset_556_2021-09-30_16-01-39.zip', 'GNEL_10_GNEL1010_20210925_-_23ad7d8b-982a-408f-92f0-3bb535f1dbc0.xml', 'Go-Ahead_Chester-le-Street_Peterlee_20210306_9', '556.0', 'Bus', 'GNEL', '10', '10', None, 'true', '7.0', '2021-09-25', None, '0.0', None, 'PB0000092/379', '10', 'Registered', 'In Scope', 'Not Seasonal', 'Stagecoach', 'No', 'No', 'Published', 'Up to date', 'No', 'No', 'Yes', 'No', 'No', 'Published', 'Not Stale', 'Yes', '18046', '10_10A-None--SYRK-HK-2026-07-26-HK_26_July_2026__SYRK_PB0000092_379_20260726-BODS_V1_1.xml', 'SYRK', '2026-06-10', '2027-06-10', '2026-07-26', None, '5357.0', 'FX_PI_01_UK_SYRK_NETWORK_FARE_Xtra‐Shfd‐STUD‐Bus_wef-20260523_20260